## Data Processing in Python Final Project

### Authors: Matyáš Tvrz, 

### To Do List:

1) download data from bezrealitky.cz and reality.idnes.cz, and from sreality.cz on other cities
2) create heatmap based on longitude and latitude - DONE
3) add property popups to map?
4) download info on airbnb prices
5) conduct a simple analysis of rental price determinants

In [6]:
# import packages

import json
import pandas as pd
import os
import requests 
import pandas as pd 
import time
import re 
import random 
import folium
from folium.plugins import HeatMap, MarkerCluster
import math

In [7]:
# get df from last request
df = pd.read_csv("df.csv")

In [9]:
# or get newest df, takes about 7 minutes
from function_scripts import request_sreality_all
df = request_sreality_all() 
df.to_csv("df.csv", index=False)

Region 1: 10 pages
Region 2: 11 pages
Region 3: 8 pages
Region 4: 21 pages
Region 5: 7 pages
Region 6: 7 pages
Region 7: 5 pages
Region 8: 13 pages
Region 9: 8 pages
Region 10: 73 pages
Region 11: 14 pages
Region 12: 26 pages
Region 13: 6 pages
Region 14: 27 pages


In [10]:
df.shape
df.head()

,labelsReleased,has_panorama,labels,is_auction,labelsAll,seo,exclusively_at_rk,category,has_floor_plan,_embedded,...,hash_id,attractive_offer,price,price_czk,_links,rus,name,region_tip,gps,has_matterport_url
0,"[[], []]",0,[],False,"[[personal, balcony, panel, cellar, elevator, ...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",1,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,1055494220,0,15000,"{'value_raw': 15000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+kk 49 m²,0,"{'lat': 48.97235750182105, 'lon': 14.483341498...",False
1,"[[], []]",0,[],False,"[[personal], [candy_shop, tavern, vet, small_s...","{'category_main_cb': 1, 'category_sub_cb': 5, ...",1,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,825790540,0,14000,"{'value_raw': 14000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+1 89 m²,0,"{'lat': 48.82766550182105, 'lon': 14.651850498...",False
2,"[[], []]",0,[],False,"[[personal, balcony, brick, elevator, not_furn...","{'category_main_cb': 1, 'category_sub_cb': 2, ...",1,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,3292041292,0,12000,"{'value_raw': 12000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 1+kk 39 m² (Jednopodlažní),0,"{'lat': 49.39005150182105, 'lon': 14.695326498...",False
3,"[[], []]",0,[],False,"[[personal, brick, cellar, elevator, parking_l...","{'category_main_cb': 1, 'category_sub_cb': 2, ...",0,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,3824505676,0,14000,"{'value_raw': 14000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 1+kk 50 m²,0,"{'lat': 48.99149950182105, 'lon': 14.459530498...",False
4,"[[], []]",0,[],False,"[[personal, brick, cellar], [small_shop, taver...","{'category_main_cb': 1, 'category_sub_cb': 5, ...",0,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,3016377164,0,12000,"{'value_raw': 12000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+1 62 m²,0,"{'lat': 48.952360501821055, 'lon': 14.30942949...",False


In [11]:
from function_scripts import get_link_and_image
df = get_link_and_image(df)
display(df)

,labelsReleased,has_panorama,labels,is_auction,labelsAll,seo,exclusively_at_rk,category,has_floor_plan,_embedded,...,price,price_czk,_links,rus,name,region_tip,gps,has_matterport_url,url,image
0,"[[], []]",0,[],False,"[[personal, balcony, panel, cellar, elevator, ...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",1,1,1,"{'favourite': {'is_favourite': False, '_links'...",...,15000,"{'value_raw': 15000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+kk 49 m²,0,"{'lat': 48.97235750182105, 'lon': 14.483341498...",False,https://www.sreality.cz/cs/v2/estates/1055494220,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...
1,"[[], []]",0,[],False,"[[personal], [candy_shop, tavern, vet, small_s...","{'category_main_cb': 1, 'category_sub_cb': 5, ...",1,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,14000,"{'value_raw': 14000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+1 89 m²,0,"{'lat': 48.82766550182105, 'lon': 14.651850498...",False,https://www.sreality.cz/cs/v2/estates/825790540,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...
2,"[[], []]",0,[],False,"[[personal, balcony, brick, elevator, not_furn...","{'category_main_cb': 1, 'category_sub_cb': 2, ...",1,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,12000,"{'value_raw': 12000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 1+kk 39 m² (Jednopodlažní),0,"{'lat': 49.39005150182105, 'lon': 14.695326498...",False,https://www.sreality.cz/cs/v2/estates/3292041292,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...
3,"[[], []]",0,[],False,"[[personal, brick, cellar, elevator, parking_l...","{'category_main_cb': 1, 'category_sub_cb': 2, ...",0,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,14000,"{'value_raw': 14000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 1+kk 50 m²,0,"{'lat': 48.99149950182105, 'lon': 14.459530498...",False,https://www.sreality.cz/cs/v2/estates/3824505676,https://d18-a.sdn.cz/d_18/c_img_p9_C/nCTUX1iX5...
4,"[[], []]",0,[],False,"[[personal, brick, cellar], [small_shop, taver...","{'category_main_cb': 1, 'category_sub_cb': 5, ...",0,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,12000,"{'value_raw': 12000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+1 62 m²,0,"{'lat': 48.952360501821055, 'lon': 14.30942949...",False,https://www.sreality.cz/cs/v2/estates/3016377164,https://d18-a.sdn.cz/d_18/c_img_p9_C/kcHp2YdDt...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13888,"[[after_reconstruction, panel, cellar], []]",0,"[Po rekonstrukci, Panelová, Sklep]",False,"[[personal, after_reconstruction, panel, cella...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",1,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,14000,"{'value_raw': 14000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+kk 57 m²,0,"{'lat': 48.743837501821055, 'lon': 16.86837749...",False,https://www.sreality.cz/cs/v2/estates/376115788,https://d18-a.sdn.cz/d_18/c_img_qA_E/kcHp2YdDt...
13889,"[[partly_furnished], []]",0,[Částečně vybavený],False,"[[personal, brick, partly_furnished], [tavern,...","{'category_main_cb': 1, 'category_sub_cb': 4, ...",0,1,0,"{'favourite': {'is_favourite': False, '_links'...",...,13000,"{'value_raw': 13000, 'unit': 'za měsíc', 'name...",{'dynamicDown': [{'href': 'https://d18-a.sdn.c...,False,Pronájem bytu 2+kk 88 m²,0,"{'lat': 48.76795850182105, 'lon': 16.214314498...",False,https://www.sreality.cz/cs/v2/estates/3755356748,https://d18-a.sdn.cz/d_18/c_img_oa_B/nsLxLojIr...
13890,"[[new_building, terrace, cellar], []]",0,"[Novostavba, Terasa, Sklep]",False,"[[new_building, personal, terrace, cellar, ele...","{'category_main_cb': 1, 'category_sub_cb': 6

In [12]:
columns_to_keep = ['locality', 'price', 'name', 'gps','hash_id','exclusively_at_rk','url','image']
df_clean = df[columns_to_keep].copy()
df_clean['flat_type'] = df_clean.name.apply(lambda x: x.split()[2])

def name_to_area(nm):
    splitted_str = nm.split()
    m2_idx = splitted_str.index('m²')
    return int(splitted_str[m2_idx - 1])


df_clean['area'] = df_clean.name.apply(name_to_area)
df_clean.head()

,locality,price,name,gps,hash_id,exclusively_at_rk,url,image,flat_type,area
0,"České Budějovice - České Budějovice 3, okres Č...",15000,Pronájem bytu 2+kk 49 m²,"{'lat': 48.97235750182105, 'lon': 14.483341498...",1055494220,1,https://www.sreality.cz/cs/v2/estates/1055494220,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...,2+kk,49
1,"Trhové Sviny, okres České Budějovice",14000,Pronájem bytu 2+1 89 m²,"{'lat': 48.82766550182105, 'lon': 14.651850498...",825790540,1,https://www.sreality.cz/cs/v2/estates/825790540,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...,2+1,89
2,Tábor,12000,Pronájem bytu 1+kk 39 m² (Jednopodlažní),"{'lat': 49.39005150182105, 'lon': 14.695326498...",3292041292,1,https://www.sreality.cz/cs/v2/estates/3292041292,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...,1+kk,39
3,"České Budějovice - České Budějovice 2, okres Č...",14000,Pronájem bytu 1+kk 50 m²,"{'lat': 48.99149950182105, 'lon': 14.459530498...",3824505676,0,https://www.sreality.cz/cs/v2/estates/3824505676,https://d18-a.sdn.cz/d_18/c_img_p9_C/nCTUX1iX5...,1+kk,50
4,"Jankov, okres České Budějovice",12000,Pronájem bytu 2+1 62 m²,"{'lat': 48.952360501821055, 'lon': 14.30942949...",3016377164,0,https://www.sreality.cz/cs/v2/estates/3016377164,https://d18-a.sdn.cz/d_18/c_img_p9_C/kcHp2YdDt...,2+1,62


In [13]:
df_clean[['lat', 'lon']] = df_clean.gps.apply(lambda x: pd.Series({'lat': x['lat'], 'lon': x['lon']}))
df_clean.head()

,locality,price,name,gps,hash_id,exclusively_at_rk,url,image,flat_type,area,lat,lon
0,"České Budějovice - České Budějovice 3, okres Č...",15000,Pronájem bytu 2+kk 49 m²,"{'lat': 48.97235750182105, 'lon': 14.483341498...",1055494220,1,https://www.sreality.cz/cs/v2/estates/1055494220,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...,2+kk,49,48.972358,14.483341
1,"Trhové Sviny, okres České Budějovice",14000,Pronájem bytu 2+1 89 m²,"{'lat': 48.82766550182105, 'lon': 14.651850498...",825790540,1,https://www.sreality.cz/cs/v2/estates/825790540,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...,2+1,89,48.827666,14.651850
2,Tábor,12000,Pronájem bytu 1+kk 39 m² (Jednopodlažní),"{'lat': 49.39005150182105, 'lon': 14.695326498...",3292041292,1,https://www.sreality.cz/cs/v2/estates/3292041292,https://d18-a.sdn.cz/d_18/c_img_qB_B/nCTUX1iX5...,1+kk,39,49.390052,14.695326
3,"České Budějovice - České Budějovice 2, okres Č...",14000,Pronájem bytu 1+kk 50 m²,"{'lat': 48.99149950182105, 'lon': 14.459530498...",3824505676,0,https://www.sreality.cz/cs/v2/estates/3824505676,https://d18-a.sdn.cz/d_18/c_img_p9_C/nCTUX1iX5...,1+kk,50,48.991500,14.459530
4,"Jankov, okres České Budějovice",12000,Pronájem bytu 2+1 62 m²,"{'lat': 48.952360501821055, 'lon': 14.30942949...",3016377164,0,https://www.sreality.cz/cs/v2/estates/3016377164,https://d18-a.sdn.cz/d_18/c_img_p9_C/kcHp2YdDt...,2+1,62,48.952361,14.309429


In [14]:
df_heatmap = df_clean[['lat', 'lon', 'price']].copy()
df_property = df_clean[['lat', 'lon', 'price','locality', 'flat_type', 'area', 'url', 'image']].copy()

In [15]:
# base map, centered on CZ
m = folium.Map(location=(49.75, 15.40), zoom_start = 8)

# heatmap layer
heat_layer = folium.FeatureGroup(name="Heat Map", show=True)

HeatMap(
    df_heatmap,
    min_opacity=0.4,
    blur=18
).add_to(heat_layer)

heat_layer.add_to(m)

# property popups layer
property_layer = folium.FeatureGroup(
    name="Properties",
    show=False
)

marker_cluster = MarkerCluster().add_to(property_layer)

# create popups with variables from df_property
for _, row in df_property.iterrows():

    popup_html = f"""
    <b>{row['price']:,} CZK</b><br>
    {row['locality']}<br>
    {row['flat_type']}<br>
    {row['area']} m²<br><br>
    <img src="{row['image']}" width="200"><br>
    <a href="{row['url']}" target="_blank">Open listing</a>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        fill=True,
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(marker_cluster)

property_layer.add_to(m)

# layer control
folium.LayerControl(collapsed=False).add_to(m)

# save
m.save("heatmap.html")